In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import eigh
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings("ignore")
import os
os.getcwd()

'C:\\Users\\vivek'

In [2]:
def yield_to_price(yield_series, maturity, face_value=1000):
    y = yield_series / 100.0
    return face_value / ((1 + y) ** maturity)

def prepare_data(path):
    data = pd.read_csv(path)
    if 'Unnamed: 0' in data.columns:
        data = data.drop(columns=['Unnamed: 0'])
    df = data.copy()
    df['USGG3YR Yield']  = yield_to_price(df['USGG3YR Yield'], 3)
    df['USGG10YR Yield'] = yield_to_price(df['USGG10YR Yield'], 10)
    df['Date'] = pd.to_datetime(df['Date'])
    value_cols = df.columns.drop('Date')
    log_returns = np.log(df[value_cols] / df[value_cols].shift(1))
    log_returns.columns = ["lr_" + c for c in value_cols]
    df = pd.concat([df['Date'], log_returns], axis=1)
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


In [5]:
df = prepare_data("Merged_data_daily.csv")

print(f"Dataset: {df.shape[0]} days × {df.shape[1]-1} assets")
print(f"Date range: {df['Date'].iloc[0].date()} to {df['Date'].iloc[-1].date()}")

Dataset: 2776 days × 474 assets
Date range: 2015-01-05 to 2026-02-19


In [6]:
def raw_cov(X):
    return pd.DataFrame(
        np.cov(X.values, rowvar=False),
        index=X.columns, columns=X.columns
    )

def ledoit_wolf_cc(X):
    n, p = X.shape
    S = np.cov(X.values, rowvar=False)
    std = np.sqrt(np.diag(S))
    corr = S / np.outer(std, std)
    rho_bar = (corr.sum() - p) / (p * (p - 1))
    F = rho_bar * np.outer(std, std)
    np.fill_diagonal(F, np.diag(S))
    var_S = sum(
        np.linalg.norm(S - np.outer(X.iloc[t], X.iloc[t]), 'fro')**2
        for t in range(n)
    ) / n
    alpha = var_S / (np.linalg.norm(S - F, 'fro')**2)
    alpha = min(alpha, 1)
    cov = (1 - alpha) * S + alpha * F
    return pd.DataFrame(cov, index=X.columns, columns=X.columns)

def marchenko_pastur_denoise(X):
    n, p = X.shape
    S = np.cov(X.values, rowvar=False)
    q = n / p
    var = np.trace(S) / p
    lambda_plus = var * (1 + np.sqrt(1/q))**2
    eigval, eigvec = eigh(S)
    noise_mask = eigval <= lambda_plus
    eigval[noise_mask] = eigval[noise_mask].mean()
    cov = eigvec @ np.diag(eigval) @ eigvec.T
    return pd.DataFrame(cov, index=X.columns, columns=X.columns)

In [7]:
def correlation_distance(corr):
    dist = np.sqrt((1 - corr) / 2.0)
    np.fill_diagonal(dist, 0)
    return dist

def cluster_assets(corr, method='ward'):
    dist = correlation_distance(corr)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method=method)
    return Z

def quasi_diagonalize(Z, n):
    return list(leaves_list(Z))

def get_cluster_var(cov, cluster_items):
    cov_slice = cov[np.ix_(cluster_items, cluster_items)]
    ivp_w = 1.0 / np.diag(cov_slice)
    ivp_w /= ivp_w.sum()
    cluster_var = float(ivp_w @ cov_slice @ ivp_w)
    return cluster_var

In [8]:
def recursive_bisection(cov, sorted_idx):
    n = len(sorted_idx)
    w = np.ones(n)
    clusters = [list(range(n))]

    while len(clusters) > 0:
        new_clusters = []
        for cluster in clusters:
            if len(cluster) <= 1:
                continue

            mid = len(cluster) // 2
            left = cluster[:mid]
            right = cluster[mid:]

            left_items  = [sorted_idx[i] for i in left]
            right_items = [sorted_idx[i] for i in right]

            var_left  = get_cluster_var(cov, left_items)
            var_right = get_cluster_var(cov, right_items)

            alpha = 1.0 - var_left / (var_left + var_right)

            for i in left:
                w[i] *= alpha
            for i in right:
                w[i] *= (1.0 - alpha)

            if len(left) > 1:
                new_clusters.append(left)
            if len(right) > 1:
                new_clusters.append(right)

        clusters = new_clusters

    return w

In [12]:
def hrp_weights(X, cov_method="mp"):
    X_df = pd.DataFrame(X)

    if cov_method == "raw":
        cov_df = raw_cov(X_df)
    elif cov_method == "shrink":
        cov_df = ledoit_wolf_cc(X_df)
    else:
        cov_df = marchenko_pastur_denoise(X_df)

    cov = cov_df.values
    n = cov.shape[0]

    std = np.sqrt(np.diag(cov))
    std[std < 1e-10] = 1e-10
    corr = cov / np.outer(std, std)
    corr = np.clip(corr, -1, 1)

    Z = cluster_assets(corr, method='ward')
    sorted_idx = quasi_diagonalize(Z, n)

    w = recursive_bisection(cov, sorted_idx)

    w_final = np.zeros(n)
    for i, idx in enumerate(sorted_idx):
        w_final[idx] = w[i]

    return w_final / w_final.sum()

In [13]:
def rolling_backtest_hrp(df,
                        train_window=252*4,
                        holding_period=21,
                        cov_method="mp"):

    returns = df.drop(columns=['Date']).values
    dates = df['Date']
    T, N = returns.shape

    equity = [1.0]
    weights_history = []
    prev_w = None

    t = train_window

    while t < T - holding_period:
        X = returns[t - train_window : t]

        w = hrp_weights(X, cov_method=cov_method)

        weights_history.append(w)

        for j in range(holding_period):
            pnl = np.dot(w, returns[t + j])
            equity.append(equity[-1] * (1 + pnl))

        t += holding_period

    return np.array(equity), weights_history

In [ ]:
equity_hrp_mp, weights_mp = rolling_backtest_hrp(df)

plt.plot(equity_hrp_mp)
plt.title("HRP Equity Curve")
plt.show()